In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tabml.datasets

In [2]:
df = tabml.datasets.download_movielen_1m()
df.keys()

dict_keys(['users', 'movies', 'ratings'])

In [3]:
users, ratings, movies = df['users'], df['ratings'], df['movies']
print('Number of user: ', len(users['UserID']))
print('Number of movie: ', len(movies['MovieID']))

Number of user:  6040
Number of movie:  3883


In [4]:
users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6040 entries, 0 to 6039
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   UserID      6040 non-null   int64 
 1   Gender      6040 non-null   object
 2   Age         6040 non-null   int64 
 3   Occupation  6040 non-null   int64 
 4   Zip-code    6040 non-null   object
dtypes: int64(3), object(2)
memory usage: 236.1+ KB


In [5]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3883 entries, 0 to 3882
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   MovieID  3883 non-null   int64 
 1   Title    3883 non-null   object
 2   Genres   3883 non-null   object
dtypes: int64(1), object(2)
memory usage: 91.1+ KB


In [6]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000209 entries, 0 to 1000208
Data columns (total 4 columns):
 #   Column     Non-Null Count    Dtype
---  ------     --------------    -----
 0   UserID     1000209 non-null  int64
 1   MovieID    1000209 non-null  int64
 2   Rating     1000209 non-null  int64
 3   Timestamp  1000209 non-null  int64
dtypes: int64(4)
memory usage: 30.5 MB


In [7]:
# CREATING UTILITY MATRIX: USER-ITEM, ITEM-RATING.

In [8]:
# user_index_by_id = {id: idx for id,idx in enumerate(users['UserID'])}
# movie_index_by_id = {id: idx for id, idx in enumerate(movies['MovieID'])}


# print(len(user_index_by_id))
# print(len(movie_index_by_id))

In [9]:
genre_list = []
for genres in movies['Genres'].unique():
    for genre in genres.split('|'):
        if  genre not in genre_list:
            genre_list.append(genre)

genre_list

['Animation',
 "Children's",
 'Comedy',
 'Adventure',
 'Fantasy',
 'Romance',
 'Drama',
 'Action',
 'Crime',
 'Thriller',
 'Horror',
 'Sci-Fi',
 'Documentary',
 'War',
 'Musical',
 'Mystery',
 'Film-Noir',
 'Western']

In [10]:
genre_index_by_name = {name:i for i, name in enumerate(genre_list)}
movie_features = np.zeros((len(movies), len(genre_list)))

for idx, movie_genres in enumerate(movies['Genres']):
    for genre in movie_genres.split('|'):
        genre_idx = genre_index_by_name[genre]
        movie_features[idx, genre_idx] = 1

movie_features

array([[1., 1., 1., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [11]:
from tqdm import tqdm

movie_index_by_id = [movie for movie in movies['MovieID']]
user_index_by_id = [user for user in users['UserID']]

user_item = np.zeros((len(user_index_by_id), len(movie_index_by_id)))

for user_id in tqdm(ratings['UserID'].unique()):
    for movie_id in ratings[ratings['UserID'] == user_id]['MovieID']:
        movie_idx = movie_index_by_id.index(movie_id)
        user_idx = user_index_by_id.index(user_id)
        user_item[user_idx, movie_idx] = ratings[ratings['UserID'] == user_id][ratings['MovieID'] == movie_id]['Rating']
user_item

  0%|          | 0/6040 [00:00<?, ?it/s]/tmp/ipykernel_231590/3901769014.py:12: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  user_item[user_idx, movie_idx] = ratings[ratings['UserID'] == user_id][ratings['MovieID'] == movie_id]['Rating']
100%|██████████| 6040/6040 [53:28<00:00,  1.88it/s]  


array([[5., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [3., 0., 0., ..., 0., 0., 0.]])

In [26]:
# Similarity matrix: similarity between users on a specific item.

def cosine(vect1, vect2):
    numerator = np.dot(vect1, vect2)
    denominator = np.linalg.norm(vect1)*np.linalg.norm(vect2)
    return numerator/denominator

def pearson(vect1, vect2):
    return np.corrcoef(vect1, vect2)[0, 1]

def similarity(vect1, vect2, type='euclidean'):
    if type==None or type == 'euclidean':
        return cosine(vect1, vect2)
    if type == 'pearson':
        return pearson(vect1, vect2)
    else:
        print('There is that type of built-in similarity function')



In [27]:
df_example = user_item[0: 100, :]
df_example

array([[5., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [3., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])